In [0]:
import configparser

config=configparser.ConfigParser()
config.read('/Workspace/Users/krishraj2408@gmail.com/Healthcare_Fraud_detection_pipeline/config2.ini')




In [0]:
beneficary_df=spark.read\
         .format('csv').option('header',True)\
             .option('inferSchma',True)\
                 .load(config['PATHS']['beneficiary_path'])

In [0]:
inpatient_df=spark.read\
         .format('csv')\
            .option('header',True)\
                 .option('inferSchema',True)\
                     .load(config['PATHS']['inpatient_path'])

In [0]:
outpatient_df=spark.read\
         .format('csv')\
            .option('header',True)\
                 .option('inferSchema',True)\
                     .load(config['PATHS']['outpatient_path'])

In [0]:
provider_df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv(config["PATHS"]["provider_path"])
)

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name
beneficary_df = (
    beneficary_df .withColumn("ingestion_ts", current_timestamp())
)


In [0]:
for df_name in ['inpatient_df', 'outpatient_df', 'provider_df']:
    df = globals()[df_name]
    df = df.withColumn('ingestion_ts', current_timestamp())
    globals()[df_name] = df

In [0]:
from pyspark.sql.functions import col 
beneficary_df.drop(col('source_file'))

In [0]:
beneficary_df.display()

Write bronze table

In [0]:
beneficary_df.write\
    .format('delta')\
        .mode('overwrite')\
            .saveAsTable(config['TABLES']['beneficiary_table'])

In [0]:
inpatient_df.write\
    .format('delta')\
        .mode('overwrite')\
            .saveAsTable(config['TABLES']['inpatient_table'])

In [0]:
outpatient_df.write\
    .format('delta')\
        .mode('overwrite')\
            .saveAsTable(config['TABLES']['outpatient_table'])

provider_df.write\
    .format('delta')\
        .mode('overwrite')\
            .saveAsTable(config['TABLES']['provider_table'])

